In [6]:
!pip install --upgrade mediapipe opencv-python numpy matplotlib --quiet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [7]:
!pip install "tensorboard>=2.14,<3" --quiet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [8]:
# !tensorboard --logdir artifacts_3_classes/tb_logs/train

In [9]:
import os
import numpy as np
import mediapipe as mp
from pathlib import Path
from typing import List, Tuple
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

In [10]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [11]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB - for mediapipe
    image.flags.writeable = False                  # Image is no longer writeable - saves a bit of memory
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR - for opencv
    return image, results

In [12]:
def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [13]:
def extract_keypoints(results):
    pose = np.zeros(33 * 4)
    left_hand = np.zeros(21 * 3)
    right_hand = np.zeros(21 * 3)
    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark]).flatten()
    if results.left_hand_landmarks:
        left_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark]).flatten()
    if results.right_hand_landmarks:
        right_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark]).flatten()
    return np.concatenate([pose, left_hand, right_hand])

In [14]:

# ---- Config ----
DATA_DIR = Path("MP_Videos_All_keyframes_3_classes/")   # your collected keypoints live here
SEQ_LEN  = 60                # frames per sequence expected by the LSTM
SAVE_DIR = Path("artifacts_3_classes") # where to persist preprocessed arrays
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ---- Discover action classes from directory names ----
places: List[str] = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
if not places:
    raise RuntimeError(f"No action folders found in {DATA_DIR.resolve()}")
print("Detected places:", places)


Detected places: ['Bedok', 'City Hall', 'MBS']


In [15]:
import cv2
import numpy as np
import mediapipe as mp
from collections import deque
from tensorflow.keras.models import load_model

# ---- Config: set these to match your notebook / file structure ----
MODEL_PATH = "artifacts_3_classes/action_lstm.keras"   # change if you saved elsewhere
THRESHOLD  = 0.60                 # confidence threshold to accept a prediction
MAX_SENTENCE = 5                  # how many recognized labels to keep on screen
PRED_SMOOTHING = 8                # number of recent predictions to mode over


# places, SEQ_LEN are expected from earlier cells:
# places           -> np.array([...])
# SEQ_LEN   -> e.g. 60
# helper functions  -> mediapipe_detection, draw_styled_landmarks, extract_keypoints

# ---- Load trained model ----
model = load_model(MODEL_PATH)

# ---- Mediapipe (Solutions API – no .task file required) ----
mp_holistic = mp.solutions.holistic

# ---- State for streaming inference ----
sequence = deque(maxlen=SEQ_LEN)   # rolling window of keypoints
pred_history = deque(maxlen=PRED_SMOOTHING)
sentence = []
last_label = "..."  # persist last shown label even when below threshold

# ---- Video capture ----
cap = cv2.VideoCapture(0)  # 0 = default webcam; change if using external cam

if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera permissions / device index.")

# A small utility to compute the mode (most frequent) class index
def mode_index(indices):
    if len(indices) == 0:
        return None
    vals, counts = np.unique(indices, return_counts=True)
    return int(vals[np.argmax(counts)])

# Optional: text drawing helper
def draw_label_bar(image, text, y=40):
    H, W = image.shape[:2]
    cv2.rectangle(image, (0, 0), (W, y + 10), (0, 0, 0), -1)
    cv2.putText(image, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

try:
    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # 1) Mediapipe inference + draw landmarks
            image, results = mediapipe_detection(frame, holistic)
            draw_styled_landmarks(image, results)

            # Check for both hands present
            both_hands = (results.left_hand_landmarks is not None) and (results.right_hand_landmarks is not None)

            if both_hands:
                # 2) Extract keypoints and build the rolling sequence
                keypoints = extract_keypoints(results)  # must match training format/ordering
                sequence.append(keypoints)
            else:
                # Show status and skip prediction update this frame
                draw_label_bar(image, "No hands in frame", y=40)
                # Optionally, clear prediction history slowly to avoid stale labels
                # pred_history.clear()

            # 3) Only run the model when we have a full valid window and both hands detected now
            if both_hands and len(sequence) == SEQ_LEN:
                # model expects shape: (1, SEQ_LEN, feature_dim)
                res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]  # (num_classes,)
                pred_class = int(np.argmax(res))
                pred_conf  = float(res[pred_class])

                # keep history for smoothing
                pred_history.append(pred_class)
                stable_idx = mode_index(list(pred_history))

                # 4) Update "sentence" if confident & changed
                if stable_idx is not None and pred_conf >= THRESHOLD:
                    label = places[stable_idx]
                    last_label = label
                    if len(sentence) == 0 or label != sentence[-1]:
                        sentence.append(label)
                    if len(sentence) > MAX_SENTENCE:
                        sentence = sentence[-MAX_SENTENCE:]

                # 5) Draw simple probability readout (top-1) only if above threshold
                if pred_conf >= THRESHOLD:
                    top_text = f"Pred: {places[pred_class]}  |  conf: {pred_conf:.2f}"
                    draw_label_bar(image, top_text, y=40)

            # 6) Show running sentence (recognized labels) - persist last label
            display_text = (" ".join(sentence) if len(sentence) > 0 else last_label)
            cv2.putText(image, display_text, (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 220, 20), 2, cv2.LINE_AA)

            # 7) Render frame
            cv2.imshow("Real-time Sign Inference", image)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):     # press 'q' to quit
                print("Quitting...")
                break

finally:
    # ---- Cleanup ----
    print("Cleaning up...")
    cap.release()
    cv2.destroyAllWindows()


I0000 00:00:1761747981.526024 76780412 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1761747981.602691 76791079 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761747981.617456 76791087 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761747981.621436 76791079 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761747981.621596 76791084 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761747981.621670 76791087 inference_feedback_manager.cc:114] Feedback manager requ

Quitting...
Cleaning up...


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from collections import deque
from tensorflow.keras.models import load_model

# ---- Config: set these to match your notebook / file structure ----
THRESHOLD  = 0.60                 # confidence threshold to accept a prediction
MAX_SENTENCE = 5                  # how many recognized labels to keep on screen
PRED_SMOOTHING = 8                # number of recent predictions to mode over


# places, SEQ_LEN are expected from earlier cells:
# places           -> np.array([...])
# SEQ_LEN   -> e.g. 60
# helper functions  -> mediapipe_detection, draw_styled_landmarks, extract_keypoints

# ---- Mediapipe (Solutions API – no .task file required) ----
mp_holistic = mp.solutions.holistic

# ---- State for streaming inference ----
sequence = deque(maxlen=SEQ_LEN)   # rolling window of keypoints
pred_history = deque(maxlen=PRED_SMOOTHING)
sentence = []
last_label = "..."  # persist last shown label even when below threshold

# ---- Video capture ----
cap = cv2.VideoCapture(0)  # 0 = default webcam; change if using external cam

if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera permissions / device index.")

# A small utility to compute the mode (most frequent) class index
def mode_index(indices):
    if len(indices) == 0:
        return None
    vals, counts = np.unique(indices, return_counts=True)
    return int(vals[np.argmax(counts)])

# Optional: text drawing helper
def draw_label_bar(image, text, y=40):
    H, W = image.shape[:2]
    cv2.rectangle(image, (0, 0), (W, y + 10), (0, 0, 0), -1)
    cv2.putText(image, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

# Draw a fixed-size bounding box around all hand landmarks
def draw_hand_bbox(image, hand_landmarks, color=(0, 200, 255), thickness=2):
    H, W = image.shape[:2]
    
    # Get all landmark positions
    x_coords = [int(lm.x * W) for lm in hand_landmarks.landmark]
    y_coords = [int(lm.y * H) for lm in hand_landmarks.landmark]
    
    # Add padding around the hand
    padding = 50
    x_min = max(0, min(x_coords) - padding)
    x_max = min(W - 1, max(x_coords) + padding)
    y_min = max(0, min(y_coords) - padding)
    y_max = min(H - 1, max(y_coords) + padding)
    
    # Draw the bounding box
    cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color, thickness)

try:
    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # 1) Mediapipe inference
            image, results = mediapipe_detection(frame, holistic)

            # Draw hand bounding boxes
            left_present  = results.left_hand_landmarks is not None
            right_present = results.right_hand_landmarks is not None
            if left_present:
                draw_hand_bbox(image, results.left_hand_landmarks, color=(255, 0, 0))
            if right_present:
                draw_hand_bbox(image, results.right_hand_landmarks, color=(0, 0, 255))

            # Check for both hands present
            both_hands = left_present and right_present

            if both_hands:
                # 2) Extract keypoints and build the rolling sequence
                keypoints = extract_keypoints(results)  # must match training format/ordering
                sequence.append(keypoints)
            else:
                # Show status and skip prediction update this frame
                draw_label_bar(image, "No hands in frame", y=40)
                # Optionally, clear prediction history slowly to avoid stale labels
                # pred_history.clear()

            # 6) Show running sentence (recognized labels) - persist last label
            display_text = (" ".join(sentence) if len(sentence) > 0 else last_label)
            cv2.putText(image, display_text, (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 220, 20), 2, cv2.LINE_AA)

            # 7) Render frame
            cv2.imshow("Real-time Sign Inference", image)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):     # press 'q' to quit
                print("Quitting...")
                break

finally:
    # ---- Cleanup ----
    print("Cleaning up...")
    cap.release()
    cv2.destroyAllWindows()


NameError: name 'SEQ_LEN' is not defined